### Machine Learning | Assignment 2 - Supervised Learning | Exercise 4

145873 | Sezer Tetik


In [31]:
import numpy as np
import pandas as pd
import math

In [32]:
iris = pd.read_csv("iris.csv")

iris

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [33]:
iris_specs = iris.iloc[:, :4].to_numpy()
iris_species = (iris["species"] == 0).astype(int).to_numpy()

cuts = np.quantile(iris_specs, [1/3, 2/3], axis=0)

labels = np.array(["low", "medium", "high"])
categorized = np.column_stack([
    labels[np.digitize(iris_specs[:, i], cuts[:, i])]
    for i in range(iris_specs.shape[1])
])

features = np.empty(iris_specs.shape, dtype=object) # Columns 0-3 hold (value, label) pairs, column 4 the target (1 = Iris-setosa)
for i, j in np.ndindex(features.shape):
    features[i, j] = (float(iris_specs[i, j]), str(categorized[i, j]))

dataset = np.column_stack((features, iris_species))
datasets = {
    category: dataset[[label == category for _, label in dataset[:, 0]]]
    for category in ["low", "medium", "high"]
}
datasets["all"] = dataset

datasets["low"]

array([[(5.1, 'low'), (3.5, 'high'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.9, 'low'), (3.0, 'medium'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.7, 'low'), (3.2, 'high'), (1.3, 'low'), (0.2, 'low'), 1],
       [(4.6, 'low'), (3.1, 'medium'), (1.5, 'low'), (0.2, 'low'), 1],
       [(5.0, 'low'), (3.6, 'high'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.6, 'low'), (3.4, 'high'), (1.4, 'low'), (0.3, 'low'), 1],
       [(5.0, 'low'), (3.4, 'high'), (1.5, 'low'), (0.2, 'low'), 1],
       [(4.4, 'low'), (2.9, 'medium'), (1.4, 'low'), (0.2, 'low'), 1],
       [(4.9, 'low'), (3.1, 'medium'), (1.5, 'low'), (0.1, 'low'), 1],
       [(4.8, 'low'), (3.4, 'high'), (1.6, 'low'), (0.2, 'low'), 1],
       [(4.8, 'low'), (3.0, 'medium'), (1.4, 'low'), (0.1, 'low'), 1],
       [(4.3, 'low'), (3.0, 'medium'), (1.1, 'low'), (0.1, 'low'), 1],
       [(5.1, 'low'), (3.5, 'high'), (1.4, 'low'), (0.3, 'low'), 1],
       [(5.1, 'low'), (3.8, 'high'), (1.5, 'low'), (0.3, 'low'), 1],
       [(5.1, 'low'), 

In [34]:
def calculate_entropy(dataset):
    labels = dataset[:, 4]

    p_plus = len(dataset[labels == 1]) / len(labels)
    p_negative = len(dataset[labels == 0]) / len(labels)

    entropy = 0
    if p_plus > 0:
        entropy -= p_plus * math.log2(p_plus)

    if p_negative > 0:
        entropy -= p_negative * math.log2(p_negative)

    return entropy

def calculate_gain(dataset: np.ndarray, feature: int):
    entropy = calculate_entropy(dataset)
    weighted_entropy = 0

    categories = np.array([item[1] for item in dataset[:, feature]]) # Extract categorical labels for the selected feature
    for category in ["low", "medium", "high"]:
        subset = dataset[categories == category]

        if len(subset) == 0:
            continue

        weighted_entropy += (
            len(subset) / len(dataset)
        ) * calculate_entropy(subset)

    return entropy - weighted_entropy

In [35]:
for category, data in datasets.items():
    setosa = int(data[:, 4].sum())
    print(
        f"{category}:\n", 
        f"setosa: {setosa}/{len(data)} ({setosa / len(data) * 100:.2f}%)\n",
        f"entropy: {calculate_entropy(data)}\n", 
        f"gains:\n",
        f"  feature (sepal_length): {calculate_gain(data, 0)}\n",
        f"  feature (sepal_width): {calculate_gain(data, 1)}\n",
        f"  feature (petal_length): {calculate_gain(data, 2)}\n",
        f"  feature (petal_width): {calculate_gain(data, 3)}\n",
        end="\n")

low:
 setosa: 40/46 (86.96%)
 entropy: 0.5586293734521992
 gains:
   feature (sepal_length): 0.0
   feature (sepal_width): 0.4685922114940189
   feature (petal_length): 0.5586293734521992
   feature (petal_width): 0.5586293734521992

medium:
 setosa: 10/53 (18.87%)
 entropy: 0.6987003526563343
 gains:
   feature (sepal_length): 0.0
   feature (sepal_width): 0.5075390718111059
   feature (petal_length): 0.6987003526563343
   feature (petal_width): 0.6987003526563343

high:
 setosa: 0/51 (0.00%)
 entropy: 0.0
 gains:
   feature (sepal_length): 0.0
   feature (sepal_width): 0.0
   feature (petal_length): 0.0
   feature (petal_width): 0.0

all:
 setosa: 50/150 (33.33%)
 entropy: 0.9182958340544896
 gains:
   feature (sepal_length): 0.500108701590577
   feature (sepal_width): 0.2875729385841578
   feature (petal_length): 0.9182958340544896
   feature (petal_width): 0.9182958340544896



In our categories (low, medium, high, all) firstly, we have calculated entropy to be able to see the uncertainty in the dataset, in terms of surprise of getting different elements. Observations about categories' entropy as follows:

**Low: ~0.56** - This indicates a moderate level of uncertainty in the low category, suggesting that the distribution of species within this category is somewhat balanced.

**Medium: ~0.70** - This indicates a higher level of uncertainty in the medium category, suggesting that the distribution of species is more varied and less predictable.

**High: 0.00** - This indicates no uncertainty in the high category, suggesting that all instances belong to a single species, making it highly predictable.

**All: ~0.92** - This indicates a high level of uncertainty in the overall dataset, suggesting that the distribution of species across all categories is quite varied and less predictable.

So, to be able to see if feature split is effective to make entropy lower, we calculated the information gain for each feature by subtracting the weighted average entropy of the resulting subsets from the original dataset's entropy. The results are as follows:

| Metric | Low | Medium | High | All |
|---|---:|---:|---:|---:|
| **Setosa / Total** | 40 / 46 | 10 / 53 | 0 / 51 | 50 / 150 |
| **Setosa (%)** | 86.96% | 18.87% | 0.00% | 33.33% |
| **Entropy** | 0.558629 | 0.698700 | 0.000000 | 0.918296 |
| **Gain: Sepal Length** | 0.000000 | 0.000000 | 0.000000 | 0.500109 |
| **Gain: Sepal Width** | 0.468592 | 0.507539 | 0.000000 | 0.287573 |
| **Gain: Petal Length** | **0.558629** | **0.698700** | 0.000000 | **0.918296** |
| **Gain: Petal Width** | **0.558629** | **0.698700** | 0.000000 | **0.918296** |

So, we can say that since the highest information gain is achieved by splitting on Petal Length and Petal Width, these features are the most effective in reducing uncertainty in the dataset for medium category. Also, observe that, in the all category, the information gain is highest for Petal Length and Petal Width too. This suggests that these features are more informative for predicting the species of the Iris flowers compared to Sepal Length and Sepal Width.

Additionally, we observe that the information gain for both Petal Length and Petal Width in the All dataset (0.918296) is equal to the original dataset's entropy (0.918296). Since information gain is calculated by subtracting the weighted average entropy of the resulting subsets from the original entropy, this means that the remaining entropy after splitting is zero. Therefore, these features completely eliminate uncertainty in the dataset, allowing us to perfectly distinguish Iris-setosa from non-Setosa flowers based on their discretized values. This also indicates that either feature alone is sufficient to classify the examples in our dataset without requiring further splits.

Thus, we can improve chances of guessing a random examples' class by separating the dataset based on Petal Length and Petal Width, as they provide the most information gain and reduce uncertainty the most. This indicates that these features are particularly effective for constructing a decision tree that distinguishes Iris-setosa from non-Setosa flowers.

To be able to build a decision tree, we can start by splitting the dataset based on the feature with the highest information gain, which is Petal Length and Petal Width. We can create a threshold value for Petal Length and Petal Width to separate the data into two branches. For example, we can choose a threshold of 2.5 cm for Petal Length. After that, we can further split the branches based on the next feature with the highest information gain, which is Petal Width. We can choose a threshold of 1.5 cm for Petal Width. This process can be repeated until we reach a stopping criterion, such as a maximum depth or a minimum number of samples in a leaf node.